# Trade Labeler Template

Classify trades based on their price and quantity relative to the order book, then export as CSV for the Prosperity Visualizer.

**Output format**: comma-delimited CSV with columns `timestamp,price,quantity,label,color,shape`.

- `timestamp` + `price` + `quantity` together form the join key to match against trade data
- `label`: any string (each unique label gets its own toggle in the visualizer)
- `color`: hex color string like `#00ff00`
- `shape`: one of `circle`, `square`, `triangle-up`, `triangle-down`, `cross`, `diamond`

In [ ]:
import pandas as pd
import numpy as np

DAY = 0

prices = pd.read_csv(f'../ROUND1/prices_round_1_day_{DAY}.csv', sep=';')
trades = pd.read_csv(f'../ROUND1/trades_round_1_day_{DAY}.csv', sep=';')

print(f'Prices: {len(prices)} rows, Trades: {len(trades)} rows')
trades.head()

In [ ]:
# Build a lookup: for each (timestamp, product) -> best bid, best ask, mid
ob_lookup = {}
for _, row in prices.iterrows():
    key = (row['timestamp'], row['product'])
    ob_lookup[key] = {
        'best_bid': row.get('bid_price_1', np.nan),
        'best_ask': row.get('ask_price_1', np.nan),
        'mid': row.get('mid_price', np.nan),
    }

def get_ob_at(timestamp, symbol):
    """Find the most recent OB snapshot at or before this timestamp."""
    # Exact match first
    if (timestamp, symbol) in ob_lookup:
        return ob_lookup[(timestamp, symbol)]
    # Find closest prior timestamp
    product_times = sorted([t for (t, s) in ob_lookup if s == symbol and t <= timestamp], reverse=True)
    if product_times:
        return ob_lookup[(product_times[0], symbol)]
    return {'best_bid': np.nan, 'best_ask': np.nan, 'mid': np.nan}

In [ ]:
# Classify each trade
labels = []

for _, trade in trades.iterrows():
    ob = get_ob_at(trade['timestamp'], trade['symbol'])
    price = trade['price']
    qty = trade['quantity']
    best_bid = ob['best_bid']
    best_ask = ob['best_ask']

    # Classification rules — customize these!
    if not pd.isna(best_ask) and price >= best_ask:
        # Bought at or above best ask = aggressive buyer (taker)
        if qty >= 5:
            label, color, shape = 'BigTaker', '#00ff00', 'triangle-up'
        else:
            label, color, shape = 'SmallTaker', '#ffff00', 'triangle-up'
    elif not pd.isna(best_bid) and price <= best_bid:
        # Sold at or below best bid = aggressive seller (taker)
        if qty >= 5:
            label, color, shape = 'BigTaker', '#00ff00', 'triangle-down'
        else:
            label, color, shape = 'SmallTaker', '#ffff00', 'triangle-down'
    else:
        # Price between bid and ask = passive / maker
        label, color, shape = 'Maker', '#ff6600', 'square'

    labels.append({
        'timestamp': trade['timestamp'],
        'price': price,
        'quantity': qty,
        'label': label,
        'color': color,
        'shape': shape,
    })

labels_df = pd.DataFrame(labels)
print(labels_df['label'].value_counts())
labels_df.head(10)

In [ ]:
labels_df.to_csv(f'trade_labels_day{DAY}.csv', index=False)
print(f'Exported {len(labels_df)} labeled trades to trade_labels_day{DAY}.csv')